# Day 51：模型微调理论与显存精确估算（Jupyter Notebook 实战手册）

> **所属阶段**：第 5 阶段 · 第五阶段：模型微调与部署 ｜ **第 8 周**：模型微调、本地推理部署与安全合规  
> **GitLab 工单**：`STARMIND-051` ｜ **开发分支**：`feat/day51-finetune-theory-vram-calc`  
> **配套 HTML5 课件**：`courseware/day51/index.html` ｜ **独立版本快照**：`versions/day51/`

---

## 🎙️ 老陈开场旁白

早啊各位，先泡杯龙井提提神。今天正式跨进第五阶段——模型微调与部署。上周复盘会林产品拍着桌子问我：'老陈，咱能不能把 Qwen2.5-7B 微调一下，省点 API 钱，顺便把客服话术调得更像咱星辰的人？' 我当场泼了她一盆冷水：别一上来就喊微调！我在大厂见过太多团队，花两万块租 A100 训了一周，效果还不如改三行 Prompt 加个 RAG。今天上午咱先把'要不要微调'的决策账算明白，再把全量微调、LoRA、QLoRA 到底吃多少显存，写成一个精确计算器，以后去 AutoDL 或魔搭租卡，心里跟明镜似的。

### 承上启下
- **昨日基线**：昨天我们完成了第四阶段多 Agent 协作与 MCP 工具生态的周测验收，星辰智脑已经能通过调外部 API 跑通复杂业务流。但财务部和法务部反馈：通用大模型对公司内部《政企招投标术语》和《星辰ERP单据规范》老是答偏，而且按 Token 计费月底账单已经破万。
- **明日预告**：明天（Day 52）我们正式进入数据工程实战，手写清洗、去重、质量过滤与 Self-Instruct 合成流水线，给 Qwen2.5-7B 造出 500 条高质量星辰客服微调语料。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day51'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-051）

杭州星辰数科智能客服与政企标书辅助场景面临两大痛点：通用闭源 API 成本居高不下（月均 1.8 万元），且在特定话术格式、内部专有缩写理解上不稳定。团队计划评估私有化微调 Qwen2.5-7B-Instruct 的可行性与算力预算。

**今日迭代目标：**
- 构建可量化的 Prompt / RAG / Fine-Tuning 三维决策评估器，输出推荐路线与理由。
- 实现高精度 GPU 显存估算器，覆盖模型权重、梯度、AdamW 优化器状态、LoRA 旁路矩阵与 KV/激活值开销。
- 内置国内主流算力平台（AutoDL 4090 24G、A100 80G、魔搭免费实例等）推荐匹配与成本预估。

**验收标准：**
- 决策引擎针对'实时知识更新'场景推荐 RAG，针对'固定JSON风格/领域话术'推荐 LoRA 微调。
- 显存计算器对 Qwen2.5-7B 在 FP16 全量微调（>110GB）、FP16 LoRA（~18-22GB）、4-bit QLoRA（~9-12GB）的估算误差在工程可接受区间内。
- 所有核心函数具备完整单元测试且在纯 CPU 环境下 100% 通过。

---

## 📚 课堂核心技术笔记

### 1. 1. 什么时候该微调？什么时候绝对别微调？

微调的本质是通过梯度更新调整条件概率分布。它最擅长解决三类问题：(1) 强格式/强风格约束（如特定 JSON Schema、客服语气、公文行文规范）；(2) 垂直领域专业术语与推理范式（医疗、法律、金融研报）；(3) 降本增效（用微调后的 7B 小模型替代昂贵的闭源大模型完成特定子任务）。相反，如果核心诉求是让模型回答'昨天刚发布的新品价格'，请直接用 RAG。

> **⚠️ 老陈避坑**：坑点：把公司 1000 份 Word 文档直接做成 QA 塞进微调，结果模型产生严重幻觉，把 A 产品的价格背到了 B 产品头上。知识查准率永远靠 RAG 兜底。

### 2. 2. 全量微调、LoRA 与 QLoRA 的显存账本

以 Qwen2.5-7B（约 7.6B 参数）为例：BF16 全量微调光权重(15.2GB)+梯度(15.2GB)+AdamW状态(91.2GB)就超过 120GB，单张 A100 80G 都装不下，必须多卡 DeepSpeed ZeRO-3；而 BF16 LoRA（r=16, target=all-linear）可训练参数仅约 40M（0.5%），权重 15.2GB + LoRA 训练态不到 1GB + 激活值约 3~5GB，单张 24G RTX 4090 即可稳跑；QLoRA 将底座量化为 4-bit NF4，权重骤降至 ~4.5GB，12G~16G 显存也能起飞。

> **⚠️ 老陈避坑**：坑点：只算权重不算激活值和 CUDA Context（约 1~1.5GB），导致 24G 显卡跑到第 3 个 step 就 OOM。务必开启 gradient_checkpointing=True 并将 seq_len 控制在业务合理上限。

### 3. 3. 国内算力平台选型实战（AutoDL vs 魔搭 vs 阿里云PAI）

个人练习与中小团队 PoC 首选 AutoDL（按小时计费，4090 24G 约 2 元/小时，自带学术加速代理方便拉 HuggingFace/ModelScope 模型）；零预算入门可用魔搭社区（ModelScope）免费 GPU 额度；企业生产数据涉密时，优先阿里云 PAI-DSW 或内网自建双卡 4090/L40S 物理机，严禁将客户敏感脱敏前数据上传公共算力平台。

> **⚠️ 老陈避坑**：坑点：AutoDL 关机不释放数据盘也会扣少量存储费，训完模型记得把 LoRA adapter 权重下载回本地或 OSS，然后及时释放实例。


## 💻 实操步骤 1：步骤一：实现微调 vs RAG vs Prompt 企业选型决策引擎

- **目标文件**：`versions/day51/src/day51_strategy_advisor.py`
- **老陈代码旁白**：先把林产品最爱问的'到底要不要微调'变成客观代码。输入业务场景的五个维度，让决策引擎给出明确结论、风险预警和落地路线。


In [ ]:
"""
Day 51 - 模块一：微调 vs RAG vs Prompt 企业选型决策引擎
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import List


@dataclass
class ScenarioProfile:
    """企业业务场景画像"""
    name: str
    knowledge_update_freq: str  # "realtime" | "weekly" | "static"
    style_strictness: int       # 1(宽松) ~ 5(极严格话术/格式)
    labeled_samples: int        # 已有或可清洗的高质量样本条数
    need_private_deploy: bool   # 是否要求私有化部署/数据不出域
    cost_sensitivity: int       # 1(不敏感) ~ 5(极度关注推理成本)


@dataclass
class DecisionReport:
    """选型决策报告"""
    scenario_name: str
    primary_strategy: str
    secondary_strategy: str
    warnings: List[str] = field(default_factory=list)
    action_plan: List[str] = field(default_factory=list)


def advise_llm_strategy(profile: ScenarioProfile) -> DecisionReport:
    """根据业务画像输出大模型技术路线决策报告"""
    warnings: List[str] = []
    action_plan: List[str] = []

    needs_rag = profile.knowledge_update_freq in ("realtime", "weekly")
    needs_ft = (
        profile.style_strictness >= 4
        or (profile.need_private_deploy and profile.cost_sensitivity >= 4)
    )

    if needs_rag and needs_ft:
        primary = "RAG + LoRA 微调混合架构"
        secondary = "先上线 Prompt + RAG 验证业务闭环，再积累日志做 LoRA 微调"
        action_plan.append("阶段1：搭建混合检索 RAG 解决高频知识准确性")
        action_plan.append("阶段2：收集 500~2000 条线上高分对话，LoRA 微调 Qwen2.5-7B 固化话术与降本")
    elif needs_rag:
        primary = "RAG 检索增强 + 精调 Prompt"
        secondary = "暂不微调，避免知识过时导致频繁重训"
        action_plan.append("构建增量文档切分与向量/BM25混合检索链路")
        action_plan.append("通过 System Prompt 约束引用溯源与拒答机制")
    elif needs_ft:
        primary = "LoRA / QLoRA 垂直领域微调"
        secondary = "若样本不足 300 条，先用大模型 Self-Instruct 合成并人工抽检"
        action_plan.append("清洗并构建 Alpaca/ShareGPT 格式领域数据集")
        action_plan.append("使用 LLaMA-Factory 在单卡 4090 上进行 Qwen2.5-7B LoRA 微调")
    else:
        primary = "Prompt Engineering + Few-Shot"
        secondary = "直接调用国产高性价比 API（DeepSeek / 通义千问）"
        action_plan.append("编写结构化提示词与 3~5 组高质量 Few-Shot 示例")
        action_plan.append("建立自动化评估集，确认基线达标后再考虑工程优化")

    # 风险预警校验
    if needs_ft and profile.labeled_samples < 300:
        warnings.append(
            f"当前标注样本仅 {profile.labeled_samples} 条，低于微调建议起步线(500条)，"
            "强行微调极易过拟合，请先扩充或合成高质量语料。"
        )
    if profile.knowledge_update_freq == "realtime" and needs_ft:
        warnings.append(
            "检测到实时知识更新诉求：切勿将实时变动事实（如价格、库存）写死在微调数据中，必须交给 RAG。"
        )

    return DecisionReport(
        scenario_name=profile.name,
        primary_strategy=primary,
        secondary_strategy=secondary,
        warnings=warnings,
        action_plan=action_plan,
    )


if __name__ == "__main__":
    cases = [
        ScenarioProfile("星辰政企智能客服", "weekly", 5, 800, True, 5),
        ScenarioProfile("每日促销政策问答", "realtime", 2, 120, False, 3),
        ScenarioProfile("内部邮件润色助手", "static", 2, 50, False, 2),
    ]
    for c in cases:
        rep = advise_llm_strategy(c)
        print(f"=== {rep.scenario_name} ===")
        print(f"主推方案: {rep.primary_strategy}")
        print(f"备选建议: {rep.secondary_strategy}")
        if rep.warnings:
            print(f"风险预警: {rep.warnings}")
        print(f"落地步骤: {rep.action_plan}\n")


### 🔍 步骤 1 代码深度解析

这里用 `dataclass` 定义清晰的结构化业务画像 `ScenarioProfile`，将原本拍脑袋的架构争论转化为可审计的规则判断。特别注意 `warnings` 中的两条红线检查：样本量不足 300 条强训必过拟合，实时知识写进微调必出幻觉。

**预期运行输出参考：**
```text
=== 星辰政企智能客服 ===
主推方案: RAG + LoRA 微调混合架构
备选建议: 先上线 Prompt + RAG 验证业务闭环，再积累日志做 LoRA 微调
落地步骤: ['阶段1：搭建混合检索 RAG 解决高频知识准确性', '阶段2：收集 500~2000 条线上高分对话，LoRA 微调 Qwen2.5-7B 固化话术与降本']
...
```


## 💻 实操步骤 2：步骤二：实现 GPU 显存精确估算器与国内算力平台推荐器

- **目标文件**：`versions/day51/src/day51_vram_calculator.py`
- **老陈代码旁白**：决策定了要上 LoRA，下一步就是找财务批预算、去 AutoDL 租卡。下面这段代码把权重、梯度、AdamW 优化器、激活值和 CUDA 缓冲全部拆开算明白，顺便自动推荐最省钱的显卡。


In [ ]:
"""
Day 51 - 模块二：大模型微调 GPU 显存精确估算器与国内算力选型工具
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List


@dataclass
class ModelSpec:
    """主流开源大模型基础参数规格"""
    name: str
    params_billion: float   # 参数量(十亿), 如 Qwen2.5-7B 实际约 7.6B
    num_layers: int         # Transformer 层数
    hidden_size: int        # 隐藏层维度


MODEL_PRESETS: Dict[str, ModelSpec] = {
    "Qwen2.5-7B": ModelSpec("Qwen2.5-7B-Instruct", 7.6, 28, 3584),
    "Qwen2.5-14B": ModelSpec("Qwen2.5-14B-Instruct", 14.8, 48, 5120),
    "Llama3.1-8B": ModelSpec("Llama-3.1-8B-Instruct", 8.0, 32, 4096),
}

GPU_CATALOG: List[Dict[str, object]] = [
    {"name": "魔搭免费单卡 (A10/V100 16G)", "vram_gb": 16, "price_cny_per_hour": 0.0},
    {"name": "AutoDL RTX 4090 (24G)", "vram_gb": 24, "price_cny_per_hour": 2.0},
    {"name": "AutoDL A40 / L40S (48G)", "vram_gb": 48, "price_cny_per_hour": 5.5},
    {"name": "AutoDL A800 / A100 (80G)", "vram_gb": 80, "price_cny_per_hour": 9.8},
    {"name": "双卡 A800 (160G DeepSpeed)", "vram_gb": 160, "price_cny_per_hour": 19.6},
]


@dataclass
class VRAMBreakdown:
    """显存明细(单位: GB)"""
    model_name: str
    method: str
    weights_gb: float
    gradients_gb: float
    optimizer_gb: float
    activations_gb: float
    cuda_overhead_gb: float
    total_gb: float
    recommended_gpu: str
    estimated_cost_cny: float


def estimate_finetune_vram(
    model_key: str,
    method: str = "lora",           # "full" | "lora" | "qlora"
    batch_size: int = 2,
    seq_len: int = 2048,
    lora_rank: int = 16,
    gradient_checkpointing: bool = True,
    train_hours: float = 2.0,
) -> VRAMBreakdown:
    """精确估算大模型微调所需 GPU 显存(GB)及国内平台推荐"""
    if model_key not in MODEL_PRESETS:
        raise ValueError(f"不支持的模型预设: {model_key}，可选: {list(MODEL_PRESETS.keys())}")
    if method not in ("full", "lora", "qlora"):
        raise ValueError("method 必须是 'full'、'lora' 或 'qlora'")

    spec = MODEL_PRESETS[model_key]
    total_params = spec.params_billion * 1e9
    gb = 1024 ** 3

    # 1. 模型权重显存
    if method == "qlora":
        # 4-bit NF4 + 双重量化缩放常数，工程实测约 0.58 Bytes/参数
        weights_bytes = total_params * 0.58
    else:
        # BF16 / FP16: 2 Bytes/参数
        weights_bytes = total_params * 2.0

    # 2. 可训练参数比例与梯度/优化器显存
    if method == "full":
        trainable_params = total_params
        # 全量混合精度训练：BF16梯度(2B) + FP32主权重(4B) + AdamW一二阶矩(8B)
        gradients_bytes = trainable_params * 2.0
        optimizer_bytes = trainable_params * 12.0
    else:
        # LoRA 挂载全部线性层(q,k,v,o,gate,up,down)，可训练参数约 2 * 7 * L * hidden * r
        trainable_params = 2 * 7 * spec.num_layers * spec.hidden_size * lora_rank
        gradients_bytes = trainable_params * 2.0
        # QLoRA 通常搭配 paged_adamw_8bit (约 4B/可训练参数)，普通 LoRA 用 FP32 AdamW (12B)
        opt_bytes_per_param = 4.0 if method == "qlora" else 12.0
        optimizer_bytes = trainable_params * opt_bytes_per_param

    # 3. 激活值显存估算（经验工程公式，与 batch_size * seq_len * hidden * layers 成正比）
    # 开启梯度检查点(Gradient Checkpointing)时仅缓存每层输入，显存约降为原来的 28%
    raw_act_bytes = batch_size * seq_len * spec.hidden_size * spec.num_layers * 10
    act_factor = 0.28 if gradient_checkpointing else 1.0
    activations_bytes = raw_act_bytes * act_factor

    # 4. PyTorch + CUDA Context 固定缓冲开销
    cuda_overhead_gb = 1.4

    weights_gb = round(weights_bytes / gb, 2)
    gradients_gb = round(gradients_bytes / gb, 2)
    optimizer_gb = round(optimizer_bytes / gb, 2)
    activations_gb = round(activations_bytes / gb, 2)
    total_gb = round(
        weights_gb + gradients_gb + optimizer_gb + activations_gb + cuda_overhead_gb, 2
    )

    # 留出 12% 安全余量防止峰值抖动 OOM
    required_safe_gb = total_gb * 1.12
    matched_gpu = GPU_CATALOG[-1]
    for gpu in GPU_CATALOG:
        if float(gpu["vram_gb"]) >= required_safe_gb:
            matched_gpu = gpu
            break

    cost = round(float(matched_gpu["price_cny_per_hour"]) * train_hours, 2)
    return VRAMBreakdown(
        model_name=spec.name,
        method=method,
        weights_gb=weights_gb,
        gradients_gb=gradients_gb,
        optimizer_gb=optimizer_gb,
        activations_gb=activations_gb,
        cuda_overhead_gb=cuda_overhead_gb,
        total_gb=total_gb,
        recommended_gpu=str(matched_gpu["name"]),
        estimated_cost_cny=cost,
    )


if __name__ == "__main__":
    for m in ("full", "lora", "qlora"):
        res = estimate_finetune_vram("Qwen2.5-7B", method=m, batch_size=2, seq_len=2048)
        print(
            f"[{res.method.upper():5s}] 总显存={res.total_gb:6.2f} GB "
            f"(权重={res.weights_gb}G, 梯度={res.gradients_gb}G, 优化器={res.optimizer_gb}G, 激活={res.activations_gb}G) "
            f"-> 推荐: {res.recommended_gpu} (预计 {res.estimated_cost_cny} 元)"
        )


### 🔍 步骤 2 代码深度解析

1. 全量微调的杀手不是权重，而是混合精度下的 `FP32主权重+一阶/二阶动量`（共 12 Bytes/参数），7.6B 模型光优化器就吃掉 84.9GB。
2. LoRA 将可训练参数压缩到 40M 左右，梯度与优化器合计不到 1GB。
3. 选卡时特意乘以 `1.12` 的安全余量系数，这是防止训练中期因个别超长样本触发 OOM 的实战经验。

**预期运行输出参考：**
```text
[FULL ] 总显存=116.58 GB (权重=14.16G, 梯度=14.16G, 优化器=84.94G, 激活=1.92G) -> 推荐: 双卡 A800 (160G DeepSpeed) (预计 39.2 元)
[LORA ] 总显存= 17.95 GB (权重=14.16G, 梯度=0.07G, 优化器=0.40G, 激活=1.92G) -> 推荐: AutoDL RTX 4090 (24G) (预计 4.0 元)
[QLORA] 总显存=  7.60 GB (权重=4.11G, 梯度=0.07G, 优化器=0.13G, 激活=1.92G) -> 推荐: 魔搭免费单卡 (A10/V100 16G) (预计 0.0 元)
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day51/tests/test_day51.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 51 单元测试：微调决策引擎与 GPU 显存估算器"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import pytest
from src.day51_strategy_advisor import ScenarioProfile, advise_llm_strategy
from src.day51_vram_calculator import estimate_finetune_vram


def test_strategy_realtime_knowledge_prefers_rag():
    profile = ScenarioProfile("促销问答", "realtime", 2, 100, False, 3)
    rep = advise_llm_strategy(profile)
    assert "RAG" in rep.primary_strategy
    assert "微调" not in rep.primary_strategy


def test_strategy_hybrid_with_low_samples_warns():
    profile = ScenarioProfile("客服混合", "weekly", 5, 120, True, 5)
    rep = advise_llm_strategy(profile)
    assert "RAG + LoRA" in rep.primary_strategy
    assert len(rep.warnings) >= 1


def test_vram_ordering_full_gt_lora_gt_qlora():
    full = estimate_finetune_vram("Qwen2.5-7B", method="full")
    lora = estimate_finetune_vram("Qwen2.5-7B", method="lora")
    qlora = estimate_finetune_vram("Qwen2.5-7B", method="qlora")
    assert full.total_gb > 100.0
    assert 15.0 < lora.total_gb < 22.0
    assert qlora.total_gb < 12.0
    assert "4090" in lora.recommended_gpu


def test_gradient_checkpointing_reduces_activation_memory():
    with_gc = estimate_finetune_vram("Qwen2.5-7B", method="lora", gradient_checkpointing=True)
    no_gc = estimate_finetune_vram("Qwen2.5-7B", method="lora", gradient_checkpointing=False)
    assert with_gc.activations_gb < no_gc.activations_gb * 0.4


def test_invalid_model_raises():
    with pytest.raises(ValueError):
        estimate_finetune_vram("Unknown-999B", method="lora")

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：扩展 Qwen2.5-32B 模型规格并对比 QLoRA 显存

**题目要求：**
在显存计算器中新增 `Qwen2.5-32B`（32.5B 参数，64 层，hidden_size=5120）预设，计算其在 seq_len=4096、batch_size=1 下 LoRA 与 QLoRA 的显存占用并判断单卡 4090(24G) 是否放得下。

**思路提示：** 往 MODEL_PRESETS 新增一条 ModelSpec 即可，注意观察 QLoRA 在 32B 下的权重就接近 18GB。


In [ ]:
from src.day51_vram_calculator import MODEL_PRESETS, ModelSpec, estimate_finetune_vram

MODEL_PRESETS["Qwen2.5-32B"] = ModelSpec("Qwen2.5-32B-Instruct", 32.5, 64, 5120)

for method in ("lora", "qlora"):
    res = estimate_finetune_vram("Qwen2.5-32B", method=method, batch_size=1, seq_len=4096)
    fits_4090 = res.total_gb * 1.12 <= 24.0
    print(f"32B {method.upper()}: 总显存={res.total_gb}GB, 单卡4090能否运行={fits_4090}, 推荐={res.recommended_gpu}")


**答案解析与评分标准：**

32B 即使 4-bit QLoRA 权重也高达 ~17.6GB，加上激活值与 CUDA 缓冲会超过 23GB，单卡 4090 极其勉强且极易 OOM，生产建议直接上 48G A40/L40S 或 80G A800。


### 作业 2（企业实战）：企业实战：多卡 DeepSpeed ZeRO-2 / ZeRO-3 显存分摊计算

**题目要求：**
编写函数 `estimate_zero_per_gpu_vram(model_key, num_gpus, zero_stage)`，模拟 ZeRO-2（梯度+优化器按卡数均分，权重每卡各存一份）与 ZeRO-3（权重+梯度+优化器全部按卡数均分）下的单卡显存占用。

**思路提示：** 先调用 `estimate_finetune_vram(..., method='full')` 得到总量，再按 `zero_stage` 对对应分量除以 `num_gpus`。


In [ ]:
from src.day51_vram_calculator import estimate_finetune_vram


def estimate_zero_per_gpu_vram(model_key: str, num_gpus: int, zero_stage: int) -> float:
    """估算 DeepSpeed ZeRO-2 / ZeRO-3 下的单卡显存占用(GB)"""
    if num_gpus < 1 or zero_stage not in (2, 3):
        raise ValueError("num_gpus>=1 且 zero_stage 需为 2 或 3")
    base = estimate_finetune_vram(model_key, method="full", batch_size=1, seq_len=2048)
    if zero_stage == 2:
        per_gpu = base.weights_gb + (base.gradients_gb + base.optimizer_gb) / num_gpus
    else:
        per_gpu = (base.weights_gb + base.gradients_gb + base.optimizer_gb) / num_gpus
    return round(per_gpu + base.activations_gb + base.cuda_overhead_gb, 2)


if __name__ == "__main__":
    for stage in (2, 3):
        print(f"Qwen2.5-7B 4卡 ZeRO-{stage} 单卡显存: {estimate_zero_per_gpu_vram('Qwen2.5-7B', 4, stage)} GB")


**答案解析与评分标准：**

理解 ZeRO-1/2/3 的切分边界是中高级 AI 工程师面试常考点：ZeRO-1 切优化器，ZeRO-2 切优化器+梯度，ZeRO-3 连模型参数一起切，用通信带宽换单卡显存。


### 作业 3（架构进阶）：架构进阶：微调 ROI 成本回本周期计算器

**题目要求：**
假设星辰智脑目前每天消耗闭源 API 500 万 Token（综合单价 8 元/百万 Token），若改为私有化部署微调后的 Qwen2.5-7B（一次性微调算力+标注成本 3000 元，包月单卡 4090 推理服务器 1800 元/月），编写 ROI 计算函数输出多少天回本及年化节省金额。

**思路提示：** 计算每日 API 成本与每日私有化固定摊销成本的差额，注意处理每日节省 <= 0 时的无法回本边界。


In [ ]:
def calculate_finetune_roi(
    daily_tokens_million: float,
    api_price_per_million: float,
    one_time_ft_cost: float,
    monthly_gpu_hosting_cost: float,
) -> dict:
    """计算私有化微调相对闭源 API 的回本周期与年化收益"""
    daily_api_cost = daily_tokens_million * api_price_per_million
    daily_self_host_cost = monthly_gpu_hosting_cost / 30.0
    daily_saving = daily_api_cost - daily_self_host_cost

    if daily_saving <= 0:
        return {"worth_it": False, "reason": "当前调用量过低，直接调 API 更划算"}

    payback_days = round(one_time_ft_cost / daily_saving, 1)
    annual_saving = round(daily_saving * 365 - one_time_ft_cost, 2)
    return {
        "worth_it": True,
        "daily_api_cost": round(daily_api_cost, 2),
        "daily_self_host_cost": round(daily_self_host_cost, 2),
        "payback_days": payback_days,
        "annual_net_saving_cny": annual_saving,
    }


if __name__ == "__main__":
    print(calculate_finetune_roi(5.0, 8.0, 3000.0, 1800.0))


**答案解析与评分标准：**

500万 Token/天 * 8元 = 40元/天，而包月 GPU 1800元/月 = 60元/天，结果反而是直接调 API 更省钱！只有当日均 API 消耗超过 60 元（约 750 万 Token/天）或存在强数据合规要求时，私有化推理才具备经济账优势。这道题能帮你挡掉 80% 拍脑袋的私有化立项。


---

## 🎯 老陈收尾复盘

行，今天两套核心工具——微调选型决策引擎和 GPU 显存精确估算器——全都过单测了。记住老陈一句话：微调改变的是模型的'行为范式、输出风格和领域表达习惯'，不是拿来塞实时知识库的。晚上回去把不同量化精度下的 7B/14B/72B 显存对比跑一遍，明天咱开始洗数据、造语料。
